# Capítulo 7 — Ajuste fino (fine-tuning)

**Livro:** começa na página **563** do PDF.

📗 **Gabarito**: soluções dos exercícios do caderno de prática. Tente sozinho antes de olhar. Cada chamada ao modelo custa centavos de centavo.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # onde ficam ia.py e os exercícios
from ia import *               # cliente, perguntar(), conversar(), embeddings(), gasto() ...
print("Pronto. Modelo padrão:", MODELO)

### Matemática da Memória · PDF p. 589

#### 🧪 Exercício: Quanta memória um modelo precisa?

Calcule a memória (GB) para **carregar** um modelo de 7 bilhões de parâmetros em FP32, FP16, INT8 e INT4. Depois estime o **treino completo** com Adam (pesos + gradientes + 2 estados do otimizador, em FP32).

In [ ]:
bytes_por_parametro = {"FP32": 4, "FP16": 2, "INT8": 1, "INT4": 0.5}
n = 7e9
for formato, b in bytes_por_parametro.items():
    print(f"{formato}: {n * b / 1e9:6.1f} GB só para os pesos")
treino = n * (4 + 4 + 4 + 4)  # pesos + gradientes + 2 estados do Adam, todos em FP32
print(f"Treino completo (Adam, FP32): ~{treino / 1e9:.0f} GB, sem contar as ativações")

### Representações Numéricas · PDF p. 594

#### 🧪 Exercício: Precisão: FP32 × FP16

Com `numpy`, guarde o número `0.1` em `float32` e `float16` e veja o erro. Some `0.0001` mil vezes em `float16`: o resultado é 0,1?

In [ ]:
import numpy as np
for tipo in [np.float32, np.float16]:
    print(tipo.__name__, f"{float(tipo(0.1)):.10f}")
soma16 = np.float16(0)
for _ in range(1000):
    soma16 = np.float16(soma16 + np.float16(0.0001))
print("Soma em float16:", float(soma16), "(o certo seria 0.1)")

### Quantização · PDF p. 598

#### 🧪 Exercício: Quantizar um vetor para INT8

Gere 1.000 pesos aleatórios (`np.random.randn`). Quantize para INT8 com escala `max(|w|)/127`, volte para float e calcule o erro médio. Quanto de memória você economizou?

In [ ]:
import numpy as np
w = np.random.randn(1000).astype(np.float32)
escala = np.abs(w).max() / 127
q = np.round(w / escala).astype(np.int8)
w_volta = q.astype(np.float32) * escala
print("Erro médio absoluto:", float(np.abs(w - w_volta).mean()))
print(f"Memória: {w.nbytes} bytes (FP32) -> {q.nbytes} bytes (INT8)")

### Ajuste fino com eficiência de parâmetros · PDF p. 606

#### 🧪 Exercício: Quantos parâmetros o LoRA treina?

Uma matriz de pesos tem 4096 × 4096. No ajuste completo, todos os parâmetros são treinados. Com **LoRA** de posto `r`, treinamos duas matrizes `4096×r` e `r×4096`. Calcule a porcentagem treinada para r = 4, 8, 16 e 64.

In [ ]:
d = 4096
total = d * d
for r in [4, 8, 16, 64]:
    lora = d * r + r * d
    print(f"r={r:>2}: {lora:>9,} parâmetros = {100 * lora / total:.2f}% do total")

## 💰 Quanto gastei

In [ ]:
print(gasto())  # quanto esta sessão custou